# PipelineMind RAG Pipeline Evaluation

**Metrics computed:** MRR@5, NDCG@5, Recall@5, Precision@5  
**Ablation study:** dense-only vs hybrid (dense+BM25+RRF) vs hybrid+rerank  
**Latency:** p50 and p95 for each retrieval mode

Pre-requisite: run `bash scripts/ingest_fast.sh` to populate ChromaDB.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from tests.eval.rag_evaluator import RAGEvaluator
from tests.eval.eval_dataset import EVAL_DATASET

evaluator = RAGEvaluator(k=5)
print(f'ChromaDB documents: {evaluator.dense.collection.count()}')
print(f'BM25 available:     {evaluator.sparse.available}')
print(f'Eval queries:       {len(EVAL_DATASET)}')

In [ ]:
# Run ablation study across all three retrieval modes
reports = {}
for mode in ['dense_only', 'hybrid', 'hybrid_rerank']:
    print(f'Evaluating {mode}...', end=' ', flush=True)
    reports[mode] = evaluator.evaluate(mode)
    r = reports[mode]
    print(f'MRR={r.mrr:.4f}  NDCG={r.ndcg:.4f}  Recall={r.recall:.4f}  p95={r.latency_p95:.0f}ms')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

modes  = list(reports.keys())
labels = ['dense\nonly', 'hybrid\n(+BM25+RRF)', 'hybrid\n+rerank']
colors = ['#4B8BFF', '#FFD700', '#2ECC71']

metrics = {
    'MRR@5':     [reports[m].mrr       for m in modes],
    'NDCG@5':    [reports[m].ndcg      for m in modes],
    'Recall@5':  [reports[m].recall    for m in modes],
    'Precision@5':[reports[m].precision for m in modes],
}

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('PipelineMind RAG Ablation Study', fontsize=15, fontweight='bold')

for ax, (metric_name, values) in zip(axes, metrics.items()):
    bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=1.2)
    ax.set_title(metric_name, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.axhline(y=0.75, color='red', linestyle='--', alpha=0.4, label='target')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('rag_ablation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rag_ablation_metrics.png')

In [ ]:
# Latency comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(modes))
w = 0.35
p50 = [reports[m].latency_p50 for m in modes]
p95 = [reports[m].latency_p95 for m in modes]
ax.bar(x - w/2, p50, w, label='p50 latency', color='#4B8BFF')
ax.bar(x + w/2, p95, w, label='p95 latency', color='#FF6B6B')
ax.axhline(y=500, color='red', linestyle='--', alpha=0.5, label='p95 target (500ms)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Latency (ms)')
ax.set_title('Retrieval Latency by Mode', fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('rag_latency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-query MRR breakdown
hr = reports['hybrid_rerank']
query_ids = [r.query_id for r in hr.per_query]
mrr_vals  = [r.mrr for r in hr.per_query]
intents   = [r.intent for r in hr.per_query]

intent_colors = {'CODE_QA': '#4B8BFF', 'CATALOGUE': '#2ECC71', 'HEALTH': '#FFD700'}
bar_colors = [intent_colors.get(i, '#999') for i in intents]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(query_ids, mrr_vals, color=bar_colors)
ax.axhline(y=0.75, color='red', linestyle='--', alpha=0.5, label='MRR target')
ax.set_xlabel('Query ID')
ax.set_ylabel('MRR@5')
ax.set_title('Per-Query MRR@5 (hybrid+rerank)', fontweight='bold')
for intent, color in intent_colors.items():
    ax.bar([], [], color=color, label=intent)
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('rag_per_query_mrr.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table
import pandas as pd
rows = []
base_mrr = reports['dense_only'].mrr
for mode in modes:
    r = reports[mode]
    lift = (r.mrr - base_mrr) / base_mrr * 100 if base_mrr else 0
    rows.append({
        'Mode':        mode,
        'MRR@5':       f'{r.mrr:.4f}',
        'NDCG@5':      f'{r.ndcg:.4f}',
        'Recall@5':    f'{r.recall:.4f}',
        'Precision@5': f'{r.precision:.4f}',
        'p50 ms':      f'{r.latency_p50:.0f}',
        'p95 ms':      f'{r.latency_p95:.0f}',
        'MRR lift':    f'{lift:+.1f}%',
    })
df = pd.DataFrame(rows).set_index('Mode')
display(df)